# Multi-ticker — replicação em PETR4 e VALE3

Roda **dois modelos** sobre **três tickers**:

- **Dumb baseline**: XGBoost com 5 features autoregressivas (sem sentimento)
- **Stage 4 setup**: Transformer com 11 features de preço + 5 features de sentimento FinBERT-PT-BR

Tickers: ITUB4 (referência), PETR4, VALE3.

Saída: tabela 3 × 2 com **AUC + bootstrap CI 95%** em cada célula. É essa tabela que vai dizer se a tese se sustenta ou se precisa pivotar.

## Cenários após executar

| Padrão observado | Conclusão |
|---|---|
| Sentimento > baseline em **3/3** tickers, CIs separados | Achado robusto. Tese sólida. |
| Sentimento > baseline em **2/3** | Achado parcial — discutir por que falha em um ticker (provável: regime/setor). |
| Sentimento > baseline em **1/3** | Achado é específico de ITUB4 — pivotar para "sentimento funciona em bancos". |
| Sentimento ≤ baseline em **3/3**, CIs sobrepostos | Pivotar tese para *comparação de representações* (Etapa 3 vs 4), não para *previsão*. |

In [1]:
import sys, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yfinance as yf
import xgboost as xgb
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from eval_utils import walk_forward_split, evaluate_model, make_binary_target

HORIZON = 21
WINDOW = 30
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

TICKERS = {
    'ITUB4': ('ITUB4.SA', '../4.finbert-br/itub4_daily_sentiment.csv'),
    'PETR4': ('PETR4.SA', '../4.finbert-br/petr4_daily_sentiment.csv'),
    'VALE3': ('VALE3.SA', '../4.finbert-br/vale3_daily_sentiment.csv'),
}
PRICE_COLS = ['Close','Volume','return','ma7','ma21','std21','lag_1','lag_2','lag_3','lag_4','lag_5']
SENT_COLS = ['n_articles','mean_logit_pos','mean_logit_neg','mean_logit_neu','mean_sentiment']
BASELINE_FEATURES = ['return','lag_1','lag_5','Volume','std21']

## 1. Funções de pipeline

Tudo encapsulado em funções porque o mesmo pipeline roda 3 vezes.

In [2]:
def load_prices(yf_ticker, period='5y'):
    df = yf.Ticker(yf_ticker).history(period=period, auto_adjust=True).reset_index()
    df['date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
    df = df[['date','Close','Volume']].copy()
    df['return'] = df['Close'].pct_change()
    df['ma7']    = df['Close'].rolling(7).mean()
    df['ma21']   = df['Close'].rolling(21).mean()
    df['std21']  = df['Close'].rolling(21).std()
    for k in range(1, 6):
        df[f'lag_{k}'] = df['Close'].shift(k)
    return df.dropna().reset_index(drop=True)

def build_dataset(yf_ticker, sentiment_csv):
    px = load_prices(yf_ticker)
    sent = pd.read_csv(sentiment_csv, parse_dates=['date'])[['date'] + SENT_COLS]
    df = px.merge(sent, on='date', how='left').sort_values('date').reset_index(drop=True)
    df[SENT_COLS] = df[SENT_COLS].ffill().fillna(0)
    df['target'] = make_binary_target(df['Close'], horizon=HORIZON)
    return df.dropna(subset=['target']).reset_index(drop=True)

def make_windows(X, y, window=WINDOW):
    Xs, ys = [], []
    for i in range(window, len(X)):
        Xs.append(X[i-window:i]); ys.append(y[i])
    return np.array(Xs), np.array(ys)

In [3]:
def run_baseline_xgb(df):
    """Dumb baseline: XGBoost com 5 features autoregressivas."""
    train, val, test = walk_forward_split(df)
    sc = StandardScaler().fit(train[BASELINE_FEATURES])
    Xtr = sc.transform(train[BASELINE_FEATURES]); ytr = train['target'].values.astype(int)
    Xva = sc.transform(val[BASELINE_FEATURES]);   yva = val['target'].values.astype(int)
    Xte = sc.transform(test[BASELINE_FEATURES]);  yte = test['target'].values.astype(int)
    pos = (ytr==1).sum(); neg = (ytr==0).sum()
    m = xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=neg/max(pos,1), eval_metric='auc', random_state=SEED,
    )
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return evaluate_model(yte, m.predict_proba(Xte)[:,1])

In [4]:
class Stage4Transformer(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, nlayers=2, window=WINDOW):
        super().__init__()
        self.proj = nn.Linear(n_features, d_model)
        pe = torch.zeros(window, d_model)
        pos = torch.arange(0, window).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos*div); pe[:, 1::2] = torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0))
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=128, dropout=0.2, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=nlayers)
        self.head = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 1))
    def forward(self, x):
        h = self.proj(x) + self.pe[:, :x.size(1), :]
        h = self.enc(h)
        return self.head(h.mean(dim=1)).squeeze(-1)

def run_stage4_transformer(df):
    """Stage 4 setup: Transformer com 11 preço + 5 sentimento."""
    FEATURES = PRICE_COLS + SENT_COLS
    train, val, test = walk_forward_split(df)
    sc = StandardScaler().fit(train[FEATURES])
    Xtr = sc.transform(train[FEATURES]); ytr = train['target'].values.astype(int)
    Xva = sc.transform(val[FEATURES]);   yva = val['target'].values.astype(int)
    Xte = sc.transform(test[FEATURES]);  yte = test['target'].values.astype(int)

    Xtw, ytw = make_windows(Xtr, ytr)
    Xvw, yvw = make_windows(Xva, yva)
    Xew, yew = make_windows(Xte, yte)

    torch.manual_seed(SEED); np.random.seed(SEED)
    model = Stage4Transformer(n_features=len(FEATURES)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    pos = (ytw==1).sum(); neg = (ytw==0).sum()
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg/max(pos,1)], device=device, dtype=torch.float32))

    Xt_t = torch.tensor(Xtw, dtype=torch.float32, device=device)
    yt_t = torch.tensor(ytw, dtype=torch.float32, device=device)
    Xv_t = torch.tensor(Xvw, dtype=torch.float32, device=device)
    yv_t = torch.tensor(yvw, dtype=torch.float32, device=device)
    Xe_t = torch.tensor(Xew, dtype=torch.float32, device=device)

    best=float('inf'); best_state=None; bad=0; patience=15
    for epoch in range(300):
        model.train(); opt.zero_grad()
        loss = loss_fn(model(Xt_t), yt_t); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval()
        with torch.no_grad():
            vl = loss_fn(model(Xv_t), yv_t).item()
        if vl < best - 1e-4:
            best=vl; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        y_score = torch.sigmoid(model(Xe_t)).cpu().numpy()
    return evaluate_model(yew, y_score)

## 2. Loop principal — 3 tickers × 2 modelos

In [5]:
rows = []
for label, (yf_t, sent_csv) in TICKERS.items():
    print(f'\n=== {label} ===')
    df = build_dataset(yf_t, sent_csv)
    print(f'  dataset: {len(df)} dias | balance={df["target"].mean():.3f}')

    print('  treinando dumb baseline (XGBoost, 5 feat) ...')
    rb = run_baseline_xgb(df)
    print(f'    AUC = {rb["auc_str"]} | acc={rb["accuracy"]:.3f}')
    rows.append({'ticker': label, 'modelo': 'baseline_xgb', **{k: rb[k] for k in ['auc','accuracy','f1_pos','f1_neg']}, 'auc_ci': rb['auc_str'], 'n_test': rb['n_test']})

    print('  treinando Stage 4 Transformer (11 preço + 5 sentimento) ...')
    rs = run_stage4_transformer(df)
    print(f'    AUC = {rs["auc_str"]} | acc={rs["accuracy"]:.3f}')
    rows.append({'ticker': label, 'modelo': 'stage4_transformer', **{k: rs[k] for k in ['auc','accuracy','f1_pos','f1_neg']}, 'auc_ci': rs['auc_str'], 'n_test': rs['n_test']})

results_df = pd.DataFrame(rows)
results_df.to_csv('results_multi_ticker.csv', index=False)
results_df


=== ITUB4 ===
  dataset: 1207 dias | balance=0.587
  treinando dumb baseline (XGBoost, 5 feat) ...
    AUC = 0.707 [0.622, 0.788] | acc=0.324
  treinando Stage 4 Transformer (11 preço + 5 sentimento) ...
    AUC = 0.319 [0.238, 0.400] | acc=0.270

=== PETR4 ===
  dataset: 1207 dias | balance=0.627
  treinando dumb baseline (XGBoost, 5 feat) ...
    AUC = 0.596 [0.519, 0.677] | acc=0.538
  treinando Stage 4 Transformer (11 preço + 5 sentimento) ...
    AUC = 0.381 [0.286, 0.489] | acc=0.336

=== VALE3 ===
  dataset: 1207 dias | balance=0.523
  treinando dumb baseline (XGBoost, 5 feat) ...
    AUC = 0.671 [0.567, 0.769] | acc=0.445
  treinando Stage 4 Transformer (11 preço + 5 sentimento) ...
    AUC = 0.981 [0.960, 0.997] | acc=0.520


,ticker,modelo,auc,accuracy,f1_pos,f1_neg,auc_ci,n_test
0,ITUB4,baseline_xgb,0.706758,0.324176,0.016000,0.485356,"0.707 [0.622, 0.788]",182
1,ITUB4,stage4_transformer,0.319270,0.269737,0.000000,0.424870,"0.319 [0.238, 0.400]",152
2,PETR4,baseline_xgb,0.596301,0.538462,0.567010,0.505882,"0.596 [0.519, 0.677]",182
3,PETR4,stage4_transformer,0.381285,0.335526,0.000000,0.502463,"0.381 [0.286, 0.489]",152
4,VALE3,baseline_xgb,0.671042,0.445055,0.521327,0.339869,"0.671 [0.567, 0.769]",182
5,VALE3,stage4_transformer,0.981463,0.519737,0.605405,0.386555,"0.981 [0.960, 0.997]",152


## 3. Tabela pivot final + decisão

In [6]:
pivot = results_df.pivot(index='ticker', columns='modelo', values='auc_ci')
print('AUC com bootstrap CI 95%:')
print(pivot.to_string())

delta = results_df.pivot(index='ticker', columns='modelo', values='auc')
delta['delta'] = delta['stage4_transformer'] - delta['baseline_xgb']
print('\nDelta (sentimento - baseline):')
print(delta[['baseline_xgb','stage4_transformer','delta']].round(3).to_string())

wins = (delta['delta'] > 0).sum()
print(f'\nSentimento bate baseline em {wins}/3 tickers')

AUC com bootstrap CI 95%:
modelo          baseline_xgb    stage4_transformer
ticker                                            
ITUB4   0.707 [0.622, 0.788]  0.319 [0.238, 0.400]
PETR4   0.596 [0.519, 0.677]  0.381 [0.286, 0.489]
VALE3   0.671 [0.567, 0.769]  0.981 [0.960, 0.997]

Delta (sentimento - baseline):
modelo  baseline_xgb  stage4_transformer  delta
ticker                                         
ITUB4          0.707               0.319 -0.387
PETR4          0.596               0.381 -0.215
VALE3          0.671               0.981  0.310

Sentimento bate baseline em 1/3 tickers


## 4. Como ler o resultado

**Não basta o delta ser positivo** — os intervalos de confiança precisam estar (idealmente) separados. Para cada par, verifique manualmente:

- Se o limite inferior do CI do `stage4_transformer` está **acima** do limite superior do CI do `baseline_xgb` → diferença significativa.
- Se há sobreposição → consistente, mas não conclusivo.

Próximo passo após este notebook: regime split por volatilidade do IBOV no melhor dos cenários — ou pivotar a narrativa da tese se 0/3 funcionarem.